In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

In [ ]:
DATA_PATH = "/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0534/data/clean_news_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

In [ ]:
print("Dataset Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print("Data Types:")
print(df.dtypes)
print("First 5 Rows:")
display(df.head())

In [ ]:
print(df.isnull().sum())

In [ ]:
df["clean_text"] = df["clean_text"].fillna("")

print("Missing clean_text values:", df["clean_text"].isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate content:", df["content"].duplicated().sum())
print("Duplicate clean text:", df["clean_text"].duplicated().sum())

In [ ]:
label_counts = df["label"].value_counts().sort_index()

print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
plt.bar(["Fake News", "True News"], label_counts.values)
plt.xlabel("News Class")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["text_length"], bins=50)
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Text Length Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df[df["label"] == 0]["text_length"], bins=50, alpha=0.6, label="Fake News")
plt.hist(df[df["label"] == 1]["text_length"], bins=50, alpha=0.6, label="True News")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Text Length Distribution by Class")
plt.legend()
plt.show()

In [ ]:
print(df["text_length"].describe())

In [ ]:
print("Original Text:")
print(df["text"].iloc[0][:500])

print("Cleaned Text:")
print(df["clean_text"].iloc[0][:500])

In [ ]:
X = df["clean_text"]
y = df["label"]

print("Feature samples:", X.shape)
print("Target samples:", y.shape)
print("Target classes:", sorted(y.unique()))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully!")

In [ ]:
rf_model.fit(X_train_tfidf, y_train)

print("Random Forest model trained successfully!")

In [ ]:
y_pred = rf_model.predict(X_test_tfidf)

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", round(accuracy * 100, 2))

In [ ]:
precision = precision_score(y_test, y_pred)

print("Precision:", precision)
print("Precision (%):", round(precision * 100, 2))

In [ ]:
recall = recall_score(y_test, y_pred)

print("Recall:", recall)
print("Recall (%):", round(recall * 100, 2))

In [ ]:
f1 = f1_score(y_test, y_pred)

print("F1 Score:", f1)
print("F1 Score (%):", round(f1 * 100, 2))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fake", "True"]
)

disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Fake", "True"]
))

In [ ]:
y_prob = rf_model.predict_proba(X_test_tfidf)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)
print("ROC-AUC (%):", round(roc_auc * 100, 2))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest ROC Curve")
plt.legend()
plt.show()

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0534/models"

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    rf_model,
    os.path.join(MODEL_DIR, "random_forest_model.pkl")
)

joblib.dump(
    tfidf,
    os.path.join(MODEL_DIR, "random_forest_tfidf_vectorizer.pkl")
)

print("Random Forest model saved successfully!")

In [ ]:
results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "ROC_AUC": [roc_auc]
})

RESULTS_DIR = "/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0534/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

results.to_csv(
    os.path.join(RESULTS_DIR, "random_forest_results.csv"),
    index=False
)

display(results)
print("Results saved successfully!")

In [ ]:
loaded_model = joblib.load(
    os.path.join(MODEL_DIR, "random_forest_model.pkl")
)

loaded_vectorizer = joblib.load(
    os.path.join(MODEL_DIR, "random_forest_tfidf_vectorizer.pkl")
)

sample_text = X_test.iloc[0]
sample_vector = loaded_vectorizer.transform([sample_text])
sample_prediction = loaded_model.predict(sample_vector)[0]

print("Actual Label:", y_test.iloc[0])
print("Predicted Label:", sample_prediction)
print("Model verification successful!")